# 083 — Selección de modelo, costo, latencia y privacidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Por request: 1,2·0,001 + 0,25·0,005 = 0,00245 → costo =
2M·0,00245 = **$4 900/mes** (entrada $2 400, salida $2 500). Con caché: el 60 % de
la entrada pasa a mitad de precio → entrada = 2 400·(0,4 + 0,6·0,5) = 2 400·0,7 =
$1 680 → total **$4 180**, ahorro **$720/mes (~14,7 %)**. La salida no se cachea:
por eso el caché ayuda menos de lo que promete si tus respuestas son largas.

**Ejercicio 2.** Umbral = 2 100/0,0009 ≈ **2,33 M requests/mes**. Supuestos
frágiles: que la GPU tiene capacidad "sobrada" a ese volumen (picos p99), que la
calidad de ambos es equivalente, y que la operación no consume ingeniería extra —
el costo fijo real suele crecer con el tráfico.

**Ejercicio 3.** (a) **C (local)**: la residencia de datos es restricción dura;
la calidad menor se compensa con revisión humana. (b) **B**: público, sin datos
sensibles, el costo y el TTFT mandan; A solo si la marca exige calidad tope.
(c) **B con batch API** (o C si ya existe la GPU): proceso nocturno sin requisito
de latencia — optimizar puro costo por documento.

**Ejercicio 4.** El contrato del laboratorio obliga a declarar evidencia
inspeccionable y limitaciones explícitas: exactamente lo que distingue un informe
de selección honesto de una recomendación de pasillo.

In [ ]:
# Ejercicio 1
R, T_in, T_out = 2_000_000, 1200, 250
p_in, p_out = 0.001, 0.005
costo_in = R * T_in / 1000 * p_in
costo_out = R * T_out / 1000 * p_out
costo = costo_in + costo_out
costo_cache = costo_in * (0.4 + 0.6 * 0.5) + costo_out
print(f"costo=${costo:,.0f}  con_cache=${costo_cache:,.0f}  ahorro=${costo-costo_cache:,.0f}")

# Ejercicio 2
print("umbral =", round(2100 / 0.0009), "requests/mes")  # ~2.33M

# Ejercicio 4
result = run_lab("evaluation", seed=83)
assert result["kind"] == "evaluation"
assert result["evidence"] and result["limitations"]
show(result)

## Reflexión

1. ¿Por qué "costo por tarea resuelta" es la métrica correcta y comparar precios
   por token entre proveedores puede engañar en ambas direcciones?
2. Da un ejemplo concreto donde el p99 de TTFT importe más que el p50 y otro donde
   ninguna latencia importe en absoluto.
3. ¿Qué evidencia mínima te haría confiar en un LLM-judge como métrica de calidad
   de tu golden set?